In [9]:
import pandas as pd

In [10]:
RAW_DATA_DIR = "../data/raw"

books = pd.read_csv(f"{RAW_DATA_DIR}/Books.csv", low_memory=False)
ratings = pd.read_csv(f"{RAW_DATA_DIR}/Ratings.csv", low_memory=False)
users = pd.read_csv(f"{RAW_DATA_DIR}/Users.csv", low_memory=False)

In [11]:
# Clean data
# Drop duplicate ratings
print(f"Total books before cleaning: {books['ISBN'].nunique():,}")
ratings = ratings.drop_duplicates(subset=["User-ID", "ISBN"])
# Exclude books with rating == 0
ratings = ratings[ratings["Book-Rating"] > 0]
books = books[books["ISBN"].isin(ratings["ISBN"].unique())]
print(f"Total books after excluding unrated: {books['ISBN'].nunique():,}")

# Exclude books with less than 3 ratings
book_counts = ratings["ISBN"].value_counts()
ratings = ratings[ratings["ISBN"].isin(book_counts[book_counts >= 3].index)]
books = books[books["ISBN"].isin(ratings["ISBN"].unique())]
print(f"Total books after excluding low-rated: {books['ISBN'].nunique():,}")

# Exclude books with null values in important columns
books = books.dropna(subset=["Book-Title", "Book-Author"])
books = books.drop_duplicates("ISBN")

ratings = ratings[ratings["ISBN"].isin(books["ISBN"])]
print(f"Total books after cleaning: {books['ISBN'].nunique():,}")
print(f"Total ratings after cleaning: {ratings.shape[0]:,}")


Total books before cleaning: 271,360
Total books after excluding unrated: 149,836
Total books after excluding low-rated: 27,804
Total books after cleaning: 27,804
Total ratings after cleaning: 239,191


In [ ]:
# Primitive query based CF implementation

# 1. Get all users who rated this book above threshold
# 2. Get all books rated by these users above threshold
# 3. Count the number of users who rated each book above threshold
# 4. Recommend the top N books with the highest counts 
def recommend_books_by_user(book_title: str, rating_threshold: int = 7, top_n: int = 10):
    isbn = books[books["Book-Title"].str.upper() == book_title.upper()]["ISBN"].values[0]
    
    users_who_rated = ratings[
        (ratings["ISBN"] == isbn) & 
        (ratings["Book-Rating"] >= rating_threshold)
    ]["User-ID"].unique()
    
    books_rated_by_users = ratings[
        (ratings["User-ID"].isin(users_who_rated)) & 
        (ratings["Book-Rating"] >= rating_threshold) &
        (ratings["ISBN"] != isbn)
    ]
    
    book_counts = books_rated_by_users["ISBN"].value_counts()
    
    recommended_isbns = book_counts.head(top_n).index
    recommended_books = books[books["ISBN"].isin(recommended_isbns)]

    recommended_books["Count"] = recommended_books["ISBN"].map(book_counts)
    recommended_books = recommended_books.sort_values("Count", ascending=False)

    return recommended_books[["Book-Title", "Book-Author"]].reset_index(drop=True)

rb = recommend_books_by_user("The Da Vinci Code", rating_threshold=7, top_n=10)
print(f"Recommended books for 'The Da Vinci Code':\n{rb}")

Recommended books for 'The Da Vinci Code':
                                          Book-Title      Book-Author
0                                Angels &amp; Demons        Dan Brown
1                          The Lovely Bones: A Novel     Alice Sebold
2                            The Secret Life of Bees    Sue Monk Kidd
3                The Red Tent (Bestselling Backlist)    Anita Diamant
4                 The Five People You Meet in Heaven      Mitch Albom
5                          Girl with a Pearl Earring  Tracy Chevalier
6  Harry Potter and the Order of the Phoenix (Boo...    J. K. Rowling
7                                        Good in Bed  Jennifer Weiner
8                              To Kill a Mockingbird       Harper Lee
9  Harry Potter and the Sorcerer's Stone (Harry P...    J. K. Rowling


In [ ]:
# Weighted version of the above approach
def recommend_books_by_user_weighted(book_title: str, rating_threshold: int = 7, top_n: int = 10):
    isbn = books[books["Book-Title"].str.upper() == book_title.upper()]["ISBN"].values[0]
    
    users_who_rated = ratings[
        (ratings["ISBN"] == isbn) & 
        (ratings["Book-Rating"] >= rating_threshold)
    ]["User-ID"].unique()
    
    books_rated_by_users = ratings[
        (ratings["User-ID"].isin(users_who_rated)) & 
        (ratings["Book-Rating"] >= rating_threshold) &
        (ratings["ISBN"] != isbn)
    ]
    
    book_scores = books_rated_by_users.groupby("ISBN")["Book-Rating"].sum()
    
    recommended_isbns = book_scores.sort_values(ascending=False).head(top_n).index
    recommended_books = books[books["ISBN"].isin(recommended_isbns)]

    recommended_books["Score"] = recommended_books["ISBN"].map(book_scores)
    recommended_books = recommended_books.sort_values("Score", ascending=False)

    return recommended_books[["Book-Title", "Book-Author"]].reset_index(drop=True)

rb_weighted = recommend_books_by_user_weighted("The Da Vinci Code", rating_threshold=7, top_n=10)
print(f"Weighted recommended books for 'The Da Vinci Code':\n{rb_weighted}")

Weighted recommended books for 'The Da Vinci Code':
                                          Book-Title      Book-Author
0                                Angels &amp; Demons        Dan Brown
1                          The Lovely Bones: A Novel     Alice Sebold
2                            The Secret Life of Bees    Sue Monk Kidd
3                The Red Tent (Bestselling Backlist)    Anita Diamant
4                 The Five People You Meet in Heaven      Mitch Albom
5  Harry Potter and the Order of the Phoenix (Boo...    J. K. Rowling
6  Harry Potter and the Sorcerer's Stone (Harry P...    J. K. Rowling
7                          Girl with a Pearl Earring  Tracy Chevalier
8                              To Kill a Mockingbird       Harper Lee
9                                        Good in Bed  Jennifer Weiner


In [ ]:
# Recommend books by same author

# 1. Get books by same author
# 2. Sort by average rating and count of ratings
# 3. Recommend top N books
def recommend_books_by_author(book_title: str, top_n: int = 10):
    book = books[books["Book-Title"].str.upper() == book_title.upper()].iloc[0]
    author = book["Book-Author"]
    
    books_by_author = books[books["Book-Author"] == author]
    
    ratings_by_author = ratings[ratings["ISBN"].isin(books_by_author["ISBN"])]
    book_stats = ratings_by_author.groupby("ISBN")["Book-Rating"].agg(["mean", "count"])
    
    recommended_isbns = book_stats.sort_values(by=["mean", "count"], ascending=False).head(top_n).index
    recommended_books = books[books["ISBN"].isin(recommended_isbns)]

    recommended_books["AvgRating"] = recommended_books["ISBN"].map(book_stats["mean"])
    recommended_books["RatingCount"] = recommended_books["ISBN"].map(book_stats["count"])
    recommended_books = recommended_books.sort_values(by=["AvgRating", "RatingCount"], ascending=False)

    return recommended_books[["Book-Title", "Book-Author"]].reset_index(drop=True)

rb_author = recommend_books_by_author("The Da Vinci Code", top_n=10)
print(f"Recommended books by same author for 'The Da Vinci Code':\n{rb_author}")

Recommended books by same author for 'The Da Vinci Code':
                               Book-Title Book-Author
0                         Deception Point   Dan Brown
1                       The Da Vinci Code   Dan Brown
2                             Illuminati.   Dan Brown
3                     Angels &amp; Demons   Dan Brown
4  El Codigo Da Vinci / The Da Vinci Code   Dan Brown
5           Digital Fortress : A Thriller   Dan Brown
6                       Angels and Demons   Dan Brown
7                     Angels &amp; Demons   Dan Brown
8                         Deception Point   Dan Brown
9           Digital Fortress : A Thriller   Dan Brown
